# Main file 

05.11.2025

Written by: Arnav Kumar Upadhyay, Cedric Meyer adapted from MATLAB code of Marc, Jan Spindler DOI-https://doi.org/10.1021/acssynbio.4c00578

This script is the main script for the TXTL v2 model.


In [1]:
# Packages functions and data types --------
include("packages.jl") 

# Sequences 
sequence_file	    = "Ecoli_K12_MG1655_geneset_cds_hpaBC_ddc.fa"    # "lacS_background.fa"
hybrid_file		    = "lacS.txt"
seqs_to_use 	    = (1,2323)                                 # Range of sequences in Fasta file. Takes all sequences from sequence a to sequence b
num_background_genes = 0
heterologous_genes  = ["hpaBC"]# ,"ddc"]

# initialization of data containers
scale = txtl_types.init_scale()         # struct that makes it possible to influence rates by scaling them

µ = 1                    # A growth rate (0.1 to 1.8) can be chosen and the proteins of the TXTL machinery will be tuned according to that growth rate and the processivity of the RNA polymerase

## flags for efficiency factors
positive_aminoacid	= true 
anti_shinedalgarno	= false      # If no hybridization file is available put flag to false. We apply the intaRNA program's intarna function (q: aSD, t: sequence, SeedBP: 5, out: tMinE) to generate a hybridization profile along the sequence. Link to the website: https://rna.informatik.uni-freiburg.de/IntaRNA/Input.jsp;jsessionid=2AE257E91E7276283F22CD77A5A8AA03
dynamic_efficiency	= true 
save_data           = false

flags = txtl_types.flags(positive_aminoacid, anti_shinedalgarno, dynamic_efficiency)

# Data import --------
mRNA 		= to_DataFrame(readmatrix("../data/input/mRNA_concentrations.xlsx", "mRNA_concentration", "A1:B2324"))
mRNA.mRNA 	= Float64.(mRNA.mRNA);

background_genes = String.(first(mRNA.gene, num_background_genes))
genes = vcat(background_genes, heterologous_genes)
println("Number of sequences: $(length(genes))") 

mRNA = subset(mRNA, :gene => ByRow(in(genes)))


## Import information about model sp ecies and sequences 
seqs, seq_names, hyb_file_dict, species_and_seqs, idx = species_and_sequences(genes, µ, sequence_file, hybrid_file, flags, seqs_to_use)
# Are mRNA gene names in E_coli gene 


## Calculate efficiency factors 
efac_pos_aa_dict, efac_int_sd_dict, dynamic_efficiencies = efficiency_factor_initialization(seqs, seq_names, hyb_file_dict, species_and_seqs)

## calculation of initial concentrations
y0, initial_concentrations = calculation_of_initial_concentration(species_and_seqs, seqs, seq_names, scale, idx);

## calculate and load kinetic parameters
p_met, p_seq, p_dyn = parameters(initial_concentrations, species_and_seqs);

## Release factor balance
RF_balance!(y0, initial_concentrations)

# Preintegration
## Definition of mass matrix
conserved = [idx.EFG, idx.EFD] # EFG, EFTu_GDP - Conserved species, Algebraic equations are given by mass balances

M = Matrix{Float64}(I, length(y0), length(y0)) # Creates unity matrix in the size of y0

M[conserved, :] .= 0; # setting mass matrix to zero to  remove time derivatives from those variables. This converts these differential equations into algebraic constraints


tags_pre = txtl_types.data_struct_preintegration(scale, idx, species_and_seqs, flags, dynamic_efficiencies, initial_concentrations, p_met, p_seq, p_dyn, mRNA.mRNA[1])

## Integration of the DAE system without DNA in the system 
y_steady_state = preintegration(tags_pre, y0, M);


# Elongation factor balance 
EF_balance!(tags_pre, y0, y_steady_state)


# Build states vector for multiple sequences 
y0_all = combined_states_vector(tags_pre, y_steady_state, seqs, seq_names)

# include_mRNA_concentrations!(y0_all, tags_pre, seq_names, seqs, mRNA)

# Prepare per-sequence tags (for use in ODE)
local_tags_list = generate_local_tags(seqs, seq_names, tags_pre, efac_pos_aa_dict, efac_int_sd_dict, mRNA);

# Integration
M = Matrix{Float64}(I, length(y0_all), length(y0_all))

M[conserved, :] .= 0

tspan 	= (0.0, 100.0) ## Define time span for the simulation

p       = generate_p(local_tags_list)

f 		= ODEFunction(ODE_parallel_distributed!, mass_matrix = M) ## Define ODEFunction to make use of mass matrix

prob 	= ODEProblem(f, y0_all, tspan, p); ## Create an ODE problem

sol = solve(prob, FBDF(autodiff=false), reltol = 1e-8, abstol = 1e-6); ## Solve ODE system


🔧 Julia started with 4 threads
Number of sequences: 1


In [2]:
for i in sol[end]
    println(i)
end

20.065442041776336
20.08441495245314
0.011765402229386797
0.020943145550994333
2.3889999533484603
1.5446150070394027
0.6637306843760689
0.7463818107395602
0.21061899830763406
2.361917500157673
0.09469119446197172
24.154502728924523
15.794527928383367
23.67831171821118
33.05785547104019
1.2289697285377902e-6
0.40998532692557077
0.2365040717734266
3.36426489847972
0.0005784628892925569
0.000264006281126925
0.0001528894125371132
0.00013205929110925797
2.6705373411897394e-5
0.0003436721057947688
1.6729730903295083e-5
0.0001199403397876133
3.776832901533287e-5
0.0002684535448215428
0.0002773058325360847
0.0003114880876779609
0.00010355928245686178
5.9623815113732094e-5
1.7022579562001503e-5
4.3385436903642536e-5
0.0010450235036755047
2.296731002482089e-5
0.00014615656589415292
1.6787376327159956e-5
0.0002400640349932828
1.72369271185932e-5
3.27518623994609e-5
0.0012544102511604588
1.2882923484567426e-20
1.6182817370496535e-20
1.0155619458549936e-20
3.503674443901486e-5
4.077451739301873e-6


In [3]:
# @benchmark solve(prob, FBDF(autodiff=false), reltol = 1e-8, abstol = 1e-6)

In [ ]:
# Inputs to test functions
du = similar(y0_all)
du = fill!(du, 0.0);
u = sol[end];
t = tspan;

# Vectors and numbers 
@unpack y_codons_lengths, offsets, n_metab, n_seq, du_metab_all, du_codons, y_codons, results, du_seq, du_metab = p.vectors

# Generate parts of u that need to be assembled specific to the sequence   
@views u_metab =  u[1:n_metab]				                            # Save all pool species from u in a vector 

i = 1

local_tags = p.local_tags_list[i]		                            # Extraction of local tags
idx = n_metab + offsets[i]                                          # Create index to access u on the sequence specific indexes
len = y_codons_lengths[i]            

# thread-safe local views
u_codons = @view u[idx+1 : idx+len]

# assemble without allocation
u_seq = similar(u, n_metab + len)
@views u_seq[1:n_metab] .= u_metab
@views u_seq[n_metab+1:end] .= u_codons

du_seq_local = du_seq[i]					                        # Initialization of sequence specific rates vector

du = du_seq_local
fill!(du, 0.0)
u = u_seq
p = local_tags


rhs_metabolites!(du, u, p)

LoadError: UndefVarError: `y0_all` not defined in `Main`
Suggestion: check for spelling errors or missing imports.